In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import cv2
cv2.setNumThreads(1)

# --- Project Configuration ---

IMAGE_STATS_FOLDER = "/Users/researcher/Downloads/PerceptionPoll/Images/Images"


# --- Helper Functions for Parsing --

def extract_image_features(image_ids):
    rows = []

    for img_id in tqdm(image_ids, desc="Computing image features"):
        img_path = os.path.join(IMAGE_STATS_FOLDER, img_id)

        try:
            feats = compute_image_features(img_path)
            feats["image_name"] = img_id
            rows.append(feats)
        except Exception:
            # if image missing or error → fill zeros
            rows.append({"image_name": img_id})

    df = pd.DataFrame(rows).set_index("image_name")
    return df

def compute_image_features(image_path):
    # Load image
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Image not found: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    features = {}

    ## ---------------- Luminance ----------------
    lum = img_gray.astype(np.float32)
    features['luminance_mean'] = np.mean(lum)
    features['luminance_std'] = np.std(lum)
    features['luminance_p05'] = np.percentile(lum, 5)
    features['luminance_p50'] = np.percentile(lum, 50)
    features['luminance_p95'] = np.percentile(lum, 95)
    features['dynamic_range'] = features['luminance_p95'] - features['luminance_p05']
    features['frac_dark'] = np.mean(lum < 50)
    features['frac_bright'] = np.mean(lum > 205)

    ## ---------------- Color ----------------
    sat = img_hsv[:, :, 1].astype(np.float32)
    val = img_hsv[:, :, 2].astype(np.float32)
    features['sat_mean'] = np.mean(sat)
    features['sat_std'] = np.std(sat)
    features['val_mean'] = np.mean(val)
    features['val_std'] = np.std(val)

    # Colorfulness metric (Hasler & Süsstrunk)
    rg = img_rgb[:, :, 0].astype(np.float32) - img_rgb[:, :, 1].astype(np.float32)
    yb = 0.5 * (img_rgb[:, :, 0].astype(np.float32) + img_rgb[:, :, 1].astype(np.float32)) - img_rgb[:, :, 2].astype(np.float32)
    features['colorfulness'] = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3*np.sqrt(rg.mean()**2 + yb.mean()**2)

    # Warmth: Red vs Blue ratio
    R = img_rgb[:, :, 0].astype(np.float32)
    B = img_rgb[:, :, 2].astype(np.float32)

    mean_r = float(np.mean(R))
    mean_b = float(np.mean(B))

    features["warmth_rb"] = (
        (mean_r - mean_b)
        / (mean_r + mean_b + 1e-6)
    )

    ## ---------------- Sharpness ----------------
    features['laplacian_var'] = cv2.Laplacian(img_gray, cv2.CV_64F).var()

    # Tenengrad (gradient magnitude)
    gx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = np.sqrt(gx**2 + gy**2)
    features['tenengrad_mean'] = np.mean(tenengrad)

    ## ---------------- Structure ----------------
    edges = cv2.Canny(img_gray, 100, 200)
    features['edge_density'] = np.mean(edges > 0)

    # Local contrast: std of small patches
    patch_size = 16
    patches = lum[:lum.shape[0]//patch_size*patch_size, :lum.shape[1]//patch_size*patch_size]
    patches = patches.reshape(patches.shape[0]//patch_size, patch_size,
                              patches.shape[1]//patch_size, patch_size)
    patches = patches.swapaxes(1,2).reshape(-1, patch_size, patch_size)
    local_contrast = [np.std(p) for p in patches]
    features['local_contrast_mean'] = np.mean(local_contrast)

    ## ---------------- Weather-ish ----------------
    # Dark channel prior
    min_channel = img_bgr.min(axis=2)
    features['dark_channel_mean'] = np.mean(min_channel)

    # Color attenuation (red - blue gradient)
    features['color_attenuation'] = np.mean(R - B)

    # White fraction
    features['white_fraction'] = np.mean(np.all(img_rgb > 240, axis=2))

    # Specular fraction: very bright pixels with low saturation
    features['specular_fraction'] = np.mean((val > 240) & (sat < 20))

    ## ---------------- Geometry ----------------
    # Vertical luminance gradient
    vert_grad = np.mean(np.diff(lum, axis=0), axis=1)
    features['vertical_lum_gradient'] = np.mean(vert_grad)

        # ---------------- 3x3 GRID IMAGE FEATURES ----------------

    h, w = img_bgr.shape[:2]
    x_bins = np.array_split(np.arange(w), 3)
    y_bins = np.array_split(np.arange(h), 3)

    for r, ys in enumerate(y_bins, start=1):
        for c, xs in enumerate(x_bins, start=1):

            cell_bgr = img_bgr[np.ix_(ys, xs)]
            cell_rgb = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2RGB)
            cell_hsv = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2HSV)
            cell_gray = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2GRAY)

            suffix = f"_{r}_{c}"

            ## ---------------- Luminance ----------------
            lum = cell_gray.astype(np.float32)
            features[f'luminance_mean{suffix}'] = np.mean(lum)
            features[f'luminance_std{suffix}'] = np.std(lum)
            features[f'luminance_p05{suffix}'] = np.percentile(lum, 5)
            features[f'luminance_p50{suffix}'] = np.percentile(lum, 50)
            features[f'luminance_p95{suffix}'] = np.percentile(lum, 95)
            features[f'dynamic_range{suffix}'] = (
                features[f'luminance_p95{suffix}'] - features[f'luminance_p05{suffix}']
            )
            features[f'frac_dark{suffix}'] = np.mean(lum < 50)
            features[f'frac_bright{suffix}'] = np.mean(lum > 205)

            ## ---------------- Color ----------------
            sat = cell_hsv[:, :, 1].astype(np.float32)
            val = cell_hsv[:, :, 2].astype(np.float32)

            features[f'sat_mean{suffix}'] = np.mean(sat)
            features[f'sat_std{suffix}'] = np.std(sat)
            features[f'val_mean{suffix}'] = np.mean(val)
            features[f'val_std{suffix}'] = np.std(val)

            rg = cell_rgb[:, :, 0].astype(np.float32) - cell_rgb[:, :, 1].astype(np.float32)
            yb = (
                0.5 * (
                    cell_rgb[:, :, 0].astype(np.float32)
                    + cell_rgb[:, :, 1].astype(np.float32)
                )
                - cell_rgb[:, :, 2].astype(np.float32)
            )

            features[f'colorfulness{suffix}'] = (
                np.sqrt(rg.std()**2 + yb.std()**2)
                + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2)
            )

            R = cell_rgb[:, :, 0].astype(np.float32)
            B = cell_rgb[:, :, 2].astype(np.float32)

            mean_r = np.mean(R)
            mean_b = np.mean(B)

            features[f'warmth_rb{suffix}'] = (
                (mean_r - mean_b)
                / (mean_r + mean_b + 1e-6)
            )

            ## ---------------- Sharpness ----------------
            features[f'laplacian_var{suffix}'] = cv2.Laplacian(
                cell_gray, cv2.CV_64F
            ).var()

            gx = cv2.Sobel(cell_gray, cv2.CV_64F, 1, 0, ksize=3)
            gy = cv2.Sobel(cell_gray, cv2.CV_64F, 0, 1, ksize=3)
            tenengrad = np.sqrt(gx**2 + gy**2)
            features[f'tenengrad_mean{suffix}'] = np.mean(tenengrad)

            ## ---------------- Structure ----------------
            edges = cv2.Canny(cell_gray, 100, 200)
            features[f'edge_density{suffix}'] = np.mean(edges > 0)

            patch_size = 16
            if (
                cell_gray.shape[0] >= patch_size
                and cell_gray.shape[1] >= patch_size
            ):
                patches = lum[
                    :lum.shape[0]//patch_size*patch_size,
                    :lum.shape[1]//patch_size*patch_size
                ]
                patches = patches.reshape(
                    patches.shape[0]//patch_size,
                    patch_size,
                    patches.shape[1]//patch_size,
                    patch_size
                )
                patches = patches.swapaxes(1, 2).reshape(
                    -1, patch_size, patch_size
                )
                local_contrast = [np.std(p) for p in patches]
                features[f'local_contrast_mean{suffix}'] = np.mean(local_contrast)
            else:
                features[f'local_contrast_mean{suffix}'] = 0.0

            ## ---------------- Weather-ish ----------------
            min_channel = cell_bgr.min(axis=2)
            features[f'dark_channel_mean{suffix}'] = np.mean(min_channel)

            features[f'color_attenuation{suffix}'] = np.mean(R - B)

            features[f'white_fraction{suffix}'] = np.mean(
                np.all(cell_rgb > 240, axis=2)
            )

            features[f'specular_fraction{suffix}'] = np.mean(
                (val > 240) & (sat < 20)
            )

            ## ---------------- Geometry ----------------
            vert_grad = np.mean(np.diff(lum, axis=0), axis=1)
            features[f'vertical_lum_gradient{suffix}'] = np.mean(vert_grad)

    return features

# --- Main Execution Block ---

def main():
    image_ids = sorted(
        f for f in os.listdir(IMAGE_STATS_FOLDER)
        if f.lower().endswith(".jpg")
    )

    image_feature_df = extract_image_features(image_ids)
    image_feature_df.to_csv("perception_poll_image_stats_grids.csv")

main()